# Customer Churn - Exploratory Data Analysis

**Goal:** Understand churn patterns and identify key drivers of customer churn.

**Dataset:** IBM Telco Customer Churn (7,043 customers)

In [ ]:
# Setup
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from db import execute_query

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded")

## 1. Load Data from Database

In [ ]:
# Load customer data with features
query = """
SELECT 
    c.*,
    f.tenure_bucket,
    f.service_count,
    f.charges_per_tenure_month,
    f.charge_change_ratio,
    f.is_month_to_month,
    f.has_autopay,
    f.potential_complaint_flag
FROM customers c
LEFT JOIN features f ON c.customer_id = f.customer_id
"""

df = execute_query(query)
print(f"Loaded {len(df):,} customers")
print(f"Churn rate: {df['churn'].mean():.2%}")
df.head()

## 2. Overall Statistics

In [ ]:
# Summary statistics
print("=" * 60)
print("CUSTOMER CHURN ANALYSIS - KEY METRICS")
print("=" * 60)
print(f"Total Customers: {len(df):,}")
print(f"Churned Customers: {df['churn'].sum():,}")
print(f"Retained Customers: {(len(df) - df['churn'].sum()):,}")
print(f"Churn Rate: {df['churn'].mean():.2%}")
print(f"\nAverage Tenure: {df['tenure'].mean():.1f} months")
print(f"Average Monthly Charges: ${df['monthly_charges'].mean():.2f}")
print(f"Average Total Charges: ${df['total_charges'].mean():.2f}")
print(f"Average Services per Customer: {df['service_count'].mean():.2f}")
print(f"\nAutopay Adoption: {df['has_autopay'].mean():.1%}")
print(f"Month-to-Month Contracts: {df['is_month_to_month'].mean():.1%}")

## 3. Churn by Contract Type

In [ ]:
# Churn rate by contract type
contract_churn = df.groupby('contract').agg({
    'churn': ['sum', 'count', 'mean']
}).round(4)
contract_churn.columns = ['churned', 'total', 'churn_rate']
contract_churn['churn_rate_pct'] = contract_churn['churn_rate'] * 100
contract_churn = contract_churn.sort_values('churn_rate', ascending=False)

print("\nChurn Rate by Contract Type:")
print(contract_churn[['total', 'churned', 'churn_rate_pct']])

# Visualization
fig = px.bar(
    contract_churn.reset_index(), 
    x='contract', 
    y='churn_rate_pct',
    title='Churn Rate by Contract Type',
    labels={'churn_rate_pct': 'Churn Rate (%)', 'contract': 'Contract Type'},
    color='churn_rate_pct',
    color_continuous_scale='Reds'
)
fig.show()

## 4. Churn by Tenure

In [ ]:
# Churn rate by tenure bucket
tenure_churn = df.groupby('tenure_bucket').agg({
    'churn': ['sum', 'count', 'mean']
}).round(4)
tenure_churn.columns = ['churned', 'total', 'churn_rate']
tenure_churn['churn_rate_pct'] = tenure_churn['churn_rate'] * 100

# Order by tenure
tenure_order = ['0-12', '13-24', '25-48', '49+']
tenure_churn = tenure_churn.reindex(tenure_order)

print("\nChurn Rate by Tenure:")
print(tenure_churn[['total', 'churned', 'churn_rate_pct']])

# Visualization
fig = px.line(
    tenure_churn.reset_index(), 
    x='tenure_bucket', 
    y='churn_rate_pct',
    title='Churn Rate by Tenure (Months)',
    labels={'churn_rate_pct': 'Churn Rate (%)', 'tenure_bucket': 'Tenure'},
    markers=True
)
fig.show()

## 5. Churn by Service Count

In [ ]:
# Churn rate by number of services
service_churn = df.groupby('service_count').agg({
    'churn': ['sum', 'count', 'mean']
}).round(4)
service_churn.columns = ['churned', 'total', 'churn_rate']
service_churn['churn_rate_pct'] = service_churn['churn_rate'] * 100

print("\nChurn Rate by Service Count:")
print(service_churn[['total', 'churned', 'churn_rate_pct']])

# Visualization
fig = px.scatter(
    service_churn.reset_index(), 
    x='service_count', 
    y='churn_rate_pct',
    size='total',
    title='Churn Rate by Number of Services',
    labels={'churn_rate_pct': 'Churn Rate (%)', 'service_count': 'Number of Services'},
    color='churn_rate_pct',
    color_continuous_scale='Reds'
)
fig.show()

## 6. Churn by Payment Method & Autopay

In [ ]:
# Churn rate by autopay
autopay_churn = df.groupby('has_autopay').agg({
    'churn': ['sum', 'count', 'mean']
}).round(4)
autopay_churn.columns = ['churned', 'total', 'churn_rate']
autopay_churn['churn_rate_pct'] = autopay_churn['churn_rate'] * 100
autopay_churn.index = ['Manual Payment', 'Autopay']

print("\nChurn Rate by Payment Type:")
print(autopay_churn[['total', 'churned', 'churn_rate_pct']])

# Visualization
fig = px.bar(
    autopay_churn.reset_index(), 
    x='has_autopay', 
    y='churn_rate_pct',
    title='Churn Rate: Autopay vs Manual Payment',
    labels={'churn_rate_pct': 'Churn Rate (%)', 'has_autopay': 'Payment Type'},
    color='churn_rate_pct',
    color_continuous_scale='Reds'
)
fig.show()

## 7. Monthly Charges Distribution

In [ ]:
# Monthly charges by churn status
fig = px.box(
    df, 
    x='churn', 
    y='monthly_charges',
    title='Monthly Charges Distribution by Churn Status',
    labels={'churn': 'Churned', 'monthly_charges': 'Monthly Charges ($)'},
    color='churn',
    category_orders={'churn': [0, 1]}
)
fig.update_xaxes(ticktext=['Retained', 'Churned'], tickvals=[0, 1])
fig.show()

print(f"\nAvg Monthly Charges (Retained): ${df[df['churn']==0]['monthly_charges'].mean():.2f}")
print(f"Avg Monthly Charges (Churned): ${df[df['churn']==1]['monthly_charges'].mean():.2f}")

## 8. Key Insights Summary

In [ ]:
print("=" * 60)
print("KEY INSIGHTS")
print("=" * 60)
print("\n1. CONTRACT TYPE IS THE STRONGEST PREDICTOR")
print("   - Month-to-month: 42.71% churn rate (10x higher risk!)")
print("   - One year: 11.27% churn rate")
print("   - Two year: 2.83% churn rate")

print("\n2. TENURE MATTERS")
print("   - New customers (0-12 months) have highest churn")
print("   - Churn decreases significantly after 12 months")
print("   - Long-term customers (49+ months) are most stable")

print("\n3. SERVICE ADOPTION REDUCES CHURN")
print(f"   - Avg services (churned): {df[df['churn']==1]['service_count'].mean():.2f}")
print(f"   - Avg services (retained): {df[df['churn']==0]['service_count'].mean():.2f}")
print("   - More services = lower churn")

print("\n4. AUTOPAY CUSTOMERS ARE MORE LOYAL")
print(f"   - Manual payment churn: {df[df['has_autopay']==0]['churn'].mean():.1%}")
print(f"   - Autopay churn: {df[df['has_autopay']==1]['churn'].mean():.1%}")

print("\n5. HIGHER MONTHLY CHARGES CORRELATE WITH CHURN")
print(f"   - Retained customers: ${df[df['churn']==0]['monthly_charges'].mean():.2f}/month")
print(f"   - Churned customers: ${df[df['churn']==1]['monthly_charges'].mean():.2f}/month")

## 9. Next Steps

Based on this EDA, we'll:

1. **Build churn classifier** (Milestone 3) - Focus on contract type, tenure, services
2. **Survival analysis** (Milestone 4) - Estimate customer lifetime by segments
3. **CLV calculation** (Milestone 6) - Prioritize high-value at-risk customers
4. **Retention targeting** (Milestone 5 & 7) - Focus on month-to-month, new customers